## Can multi-listing (“professional”) hosts be distinguished from single-listing hosts, and what operating differences show up in the data?

# Data pre-processing

**Implemented**
- Handle missing values for bedrooms and baths
- Property type consolidation
- Host listings count -> single/multi listing flag

**Not implemented**
- Minimum-stay discretisation
- Distance-from-CBD discretisation
- Add number of listings nearby to this property
---
We choose to implement the first three tasks listed.
- The single/multi listing flag is necessary as our research question involves predicting which hosts fall into which category.
- We chose to also implement property-type consolidation as we believed this may be correlated with host types, and reducing data scarcity by having fewer, more populated buckets will allow us to draw more meaningful relationships.
- Out of the remaining possible tasks, we felt that filling the gaps in bathroom and bedroom count would be the most informative when trying to derive potential relationships with host type

In [168]:
import pandas as pd
import re

df = pd.read_csv('listings.csv')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25728 entries, 0 to 25727
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            25728 non-null  int64  
 1   listing_url                                   25728 non-null  str    
 2   scrape_id                                     25728 non-null  int64  
 3   last_scraped                                  25728 non-null  str    
 4   source                                        25728 non-null  str    
 5   name                                          25728 non-null  str    
 6   description                                   25276 non-null  str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   25728 non-null  str    
 9   host_id                                       25728 non-null  int64  
 1

In [169]:
## Single vs multi-listing count
df = df[df['host_listings_count'] != 0]
df['host_type'] = df['calculated_host_listings_count'].apply(
    lambda x: 'single-listing' if x==1 else 'multi-listing'
)

In [170]:
## Consolidate property_types into fewer categories:
# Apartment
# House
# Homestay
# Hotel
# Vehicle
# Novel
# Other

def consolidate_property_type(raw_string):
    cleaned = raw_string.lower()
    cleaned = re.sub(r'.*?\broom\b( in)? ?', '', cleaned)  # remove '___ room in...'
    cleaned = re.sub(r'^entire\s?', '', cleaned)  # remove 'Entire ...'

    consolidated_types = {
        'apartment': ['rental unit', 'serviced apartment', 'condo', 'home/apt'],  # checked home/apt
        'house': ['cabin', 'tiny home', 'earthen home', 'home',
                  'house', 'cottage', 'villa', 'chalet', 'vacation home', 'nature lodge',
                  'bungalow', 'casa particular', 'townhouse'],
        'homestay': ['bed and breakfast', 'floor', 'guest suite', 'guesthouse'],  # checked floor
        'hotel': ['boutique hotel', 'aparthotel', 'hostel', 'hotel', 'resort', 'holiday park'],
        'vehicle': ['bus', 'camper/rv', 'boat', 'train'],
        'novel': ['yurt', 'tipi', 'treehouse', 'barn', 'castle', 'tent', 'farm stay',
                  'dome', 'religious building', 'hut'],
        'other/unknown': ['loft', '', 'place', 'tower', 'minsu', 'kezhan']  # since different meanings
    }

    for key, values in consolidated_types.items():
        if cleaned in values:
            return key

    return 'other/unknown'  # fallback which shouldn't trigger with current dataset

df['consolidated_property_type'] = df['property_type'].apply(consolidate_property_type)

In [171]:
## Missing values for bedrooms
# If room type is private room or shared room, set bedrooms to 1
# Else try scraping using regex
# Otherwise leave nan

def extract_bedrooms(row):
    if not pd.isna(row['bedrooms']):
        return row['bedrooms']
    elif row['room_type'] in ["Private room", "Shared room"]:
        return 1.0
    else:
        result = re.search(r'(\d+(\.\d+)?)[ -]?(?i:bedroom|bedrooms|bdr|br)\b', str(row['description']))
        if result is not None:
            return float(result.group(1))
        else:
            return float('nan')

df['bedrooms_calculated'] = df.apply(extract_bedrooms, axis=1)


# df.loc[pd.isna(df['bedrooms']), ['bedrooms', 'room_type', 'description', 'bedrooms_calculated']]

In [172]:
## Missing values for bathrooms
# Use bathrooms column
# Else extract from bathrooms_text
# ** If fails, try extracting from description  <---- tried this, none of them work so dropped it
# Else leave nan

def extract_bathrooms(row):
    if not pd.isna(row['bathrooms']):
        return row['bathrooms']
    elif not pd.isna(row['bathrooms_text']):
        num = re.search(r'(\d+(\.\d+)?) (shared |private )?baths?', str(row['bathrooms_text']))
        halves = re.search(r'(?i:half-bath)', str(row['bathrooms_text']))
        if num is not None:
            return float(num.group(1))
        elif halves is not None:
            return 0.5
    else: 
        return float('nan')
    

df['bathrooms_calculated'] = df.apply(extract_bathrooms, axis=1)


# df.loc[df['bathrooms_calculated'].isna(), ['description', 'bathrooms', 'bathrooms_text', 'bathrooms_calculated']]